# EEE40016 — Rule-Based Phase-Aware DOE

**Notebook companion to `mihan_dynamic_doe.py`** · PSS SINCAL + Python

**Purpose:** Evaluate a third DOE strategy alongside Matthew’s *static DOE* and *SVM DOE*. This notebook runs the **already tested** SINCAL-verified, per-PV rule-based dispatch algorithm and produces reproducible summary tables and figures.



## 1 — Configurable experiment

In [ ]:
from pathlib import Path
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
print('Notebook directory:', ROOT)
print('Controller found:', (ROOT / 'mihan_dynamic_doe.py').is_file())
print('SINCAL model found:', (ROOT / 'sincal_networks' / 'LV1.sin').is_file())
assert (ROOT / 'mihan_dynamic_doe.py').is_file(), 'Put notebook beside mihan_dynamic_doe.py'
assert (ROOT / 'sincal_networks' / 'LV1.sin').is_file(), 'Expected sincal_networks/LV1.sin'

# Change these to fit experiment and the available profiles.
START = '2023-03-28 14:35'
END   = '2023-03-28 16:45'
STEP  = 1        # 1 = 5 min; 3 = 15 min on a continuous five-minute dataset
MAX_PV_KW = 5.0  # gross PV capacity ceiling used in the original rule-based test
V_MIN_PU = 0.94
V_MAX_PU = 1.10
VUF_MAX_PCT = 2.0


## 2 — Execute real SINCAL power flows

The controller (1) reads paired household profiles, (2) proposes full available PV on each physical phase, (3) solves an **unbalanced power flow**, (4) checks voltage/VUF and transformer loading if found, (5) bisects individual generator outputs if needed, and (6) writes results to its own disposable network copy.


In [ ]:
import mihan_dynamic_doe as controller
controller = importlib.reload(controller)
controller.START = START
controller.END = END
controller.STEP = STEP
controller.MAX_PV_KW = MAX_PV_KW
controller.V_MIN_PU = V_MIN_PU
controller.V_MAX_PU = V_MAX_PU
controller.VUF_MAX_PCT = VUF_MAX_PCT

controller.run()


## 3 — Validate and summarise the completed run

In [ ]:
out = ROOT / 'mihan_doe_outputs'
d = pd.read_csv(out / 'dynamic_dispatch.csv', parse_dates=['time'])
m = pd.read_csv(out / 'dynamic_metrics.csv', parse_dates=['time'])
l = pd.read_csv(out / 'dynamic_limits.csv', parse_dates=['time'])

assert len(d) and len(m) and len(l), 'Missing result rows'
assert d.groupby(['time', 'pv']).size().max() == 1, 'Duplicate time/PV rows'
assert d['pv_dispatched_kW'].le(d['pv_available_kW'] + 1e-8).all(), 'Dispatch exceeds available PV'
assert (d['curtailed_kW'] >= -1e-8).all(), 'Invalid curtailment'
assert m['converged'].all(), 'Some SINCAL solves failed'

unique_times = pd.DatetimeIndex(sorted(d.time.unique()))
if len(unique_times) < 2:
    raise ValueError('Need at least two time steps to integrate energy')
step_hours = pd.Series(unique_times).diff().dropna().dt.total_seconds() / 3600
assert np.allclose(step_hours, step_hours.iloc[0]), 'Irregular timestamps: rework energy integration'
dt_h = float(step_hours.iloc[0])

summary = pd.DataFrame([{
    'controller':'Mihan rule-based DOE',
    'n_steps':len(unique_times),
    'step_hours':dt_h,
    'pv_energy_kWh':d.pv_dispatched_kW.sum()*dt_h,
    'energy_export_kWh':d.net_export_kW.sum()*dt_h,
    'pv_curtailed_kWh':d.curtailed_kW.sum()*dt_h,
    'curtailment_pct':100*d.curtailed_kW.sum()/d.pv_available_kW.sum() if d.pv_available_kW.sum()>0 else 0,
    'max_voltage_pu':m.max_v_pu.max(),
    'min_voltage_pu':m.min_v_pu.min(),
    'max_vuf_pct':m.max_vuf_pct.max(),
    'total_power_flows':int(m.power_flow_calls.sum()),
    'steps_with_curtailment':int((d.groupby('time').curtailed_kW.sum()>1e-8).sum()),
}])
display(summary.T.rename(columns={0:'Value'}))
print('PV phases:', d[['pv','phase']].drop_duplicates().to_string(index=False))
print('Transformer checked each step:', bool(m.trafo_checked.all()))
print('Line ampacity checked: NO — cable ratings missing in initial converted model')


## 4 — Generate figures for report and presentation

In [ ]:
fig_dir = ROOT / 'mihan_doe_outputs' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(9,4.2))
for pv, group in d.groupby('pv'):
    ax.plot(group.time, group.dynamic_export_limit_kW, marker='.', label=str(pv))
ax.set(title='Rule-based DOE — net export by PV', xlabel='Time', ylabel='Net export (kW)')
ax.legend(); ax.grid(True, alpha=.3); fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(fig_dir / 'rule_based_net_export.png', dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(9,4.2))
ax.plot(m.time, m.max_v_pu, marker='.', label='Maximum bus voltage (pu)')
ax.axhline(V_MAX_PU, ls='--', color='gray', label=f'Upper threshold {V_MAX_PU:.4f} pu')
ax.set(title='Rule-based DOE — SINCAL voltage check', xlabel='Time', ylabel='Voltage (pu)')
ax.legend(); ax.grid(True, alpha=.3); fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(fig_dir / 'rule_based_max_voltage.png', dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(9,4.2))
curt = d.groupby('time', as_index=False).curtailed_kW.sum()
ax.plot(curt.time, curt.curtailed_kW, marker='.')
ax.set(title='Rule-based DOE — curtailed PV power', xlabel='Time', ylabel='Curtailment (kW)')
ax.grid(True, alpha=.3); fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(fig_dir / 'rule_based_curtailment.png', dpi=180)
plt.show()


## 5 — Export  mattdoe-compatible summary

This saves metric labels aligned with the static/SVM summary screenshot where possible. **Losses are deliberately left blank:** the first rule-based controller's output does not calculate integrated line or transformer losses. Don't invent those figures. For fair cross-controller comparisons, use the same timestamps, available PV, load profiles, ceilings, voltage/VUF/thermal limits and dispatch conventions.


In [ ]:
s = summary.iloc[0]
comparison_column = pd.Series({
    'n_steps':s.n_steps,
    'step_hours':s.step_hours,
    'line_losses_kWh':np.nan,
    'trafo_losses_kWh':np.nan,
    'energy_export_kWh':s.energy_export_kWh,
    'pv_energy_kWh':s.pv_energy_kWh,
    'pv_curtailed_kWh':s.pv_curtailed_kWh,
    'curtailment_pct':s.curtailment_pct,
}, name='rule-based DOE')
comparison_column.to_csv(out / 'rule_based_comparison_column.csv', header=True)
summary.to_csv(out / 'rule_based_summary.csv', index=False)
print(comparison_column.to_string())
print('Saved:', out / 'rule_based_summary.csv')
print('Saved:', out / 'rule_based_comparison_column.csv')
